In [ ]:
# セル1: 認証とデータ読み込み
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
import pandas as pd

creds, _ = default()
gc = gspread.authorize(creds)

# スプレッドシート名を指定
SHEET_NAME = 'バレー スタッツ'

ss = gc.open(SHEET_NAME)
ws = ss.worksheet('生データ')
df = pd.DataFrame(ws.get_all_records())

print(f'全{len(df)}件')
print(f'日付: {df["date"].unique()}')
print(f'相手: {df["opponent"].unique()}')
df.head()

# セル2: 基本集計
!pip install japanize-matplotlib -q
import matplotlib.pyplot as plt
import japanize_matplotlib
import seaborn as sns

# 自チームの得点・ミス
us_plays = df[df['team']=='自チーム']
print('=== 自チーム プレー別 ===')
print(pd.crosstab(us_plays['play_type'], us_plays['result']))

# セル3: 選手別スタッツ
us_points = df[(df['team']=='自チーム') & (df['result']=='得点')]
us_miss = df[(df['team']=='自チーム') & (df['result']=='ミス')]

stats = pd.DataFrame({
    '得点': us_points.groupby('player').size(),
    'ミス': us_miss.groupby('player').size(),
}).fillna(0).astype(int)
stats['合計'] = stats['得点'] + stats['ミス']
stats['得点率'] = (stats['得点'] / stats['合計'] * 100).round(1)
stats = stats.sort_values('得点', ascending=False)
print(stats)

# セル4: スコア推移グラフ
for s in df['set'].unique():
    sd = df[df['set']==s].reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(12,3))
    ax.plot(sd.index, sd['score_us'], 'b-o', markersize=3, label='自チーム')
    ax.plot(sd.index, sd['score_them'], 'r-o', markersize=3, label='相手')
    ax.set_title(f'SET {s} スコア推移')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# セル5: サイドアウト率・ブレイク率
so = df[(df['serve_team']!=df['point_team'].map(lambda x: '自チーム' if x=='自チーム' else '相手'))]
# 相手サーブ時に自チームが得点 = サイドアウト
opp_serve = df[df['serve_team']!='自チーム']
sideout = opp_serve[opp_serve['point_team']=='自チーム']
so_rate = len(sideout)/len(opp_serve)*100 if len(opp_serve)>0 else 0

# 自チームサーブ時に自チームが得点 = ブレイク
us_serve = df[df['serve_team']=='自チーム']
brk = us_serve[us_serve['point_team']=='自チーム']
brk_rate = len(brk)/len(us_serve)*100 if len(us_serve)>0 else 0

print(f'サイドアウト率: {so_rate:.1f}% ({len(sideout)}/{len(opp_serve)})')
print(f'ブレイク率: {brk_rate:.1f}% ({len(brk)}/{len(us_serve)})')

# セル6: ローテーション別分析
rot_stats = df.groupby('rotation').apply(lambda g: pd.Series({
    '自チーム得点': (g['point_team']=='自チーム').sum(),
    '相手得点': (g['point_team']!='自チーム').sum(),
}))
rot_stats['得点率'] = (rot_stats['自チーム得点'] / (rot_stats['自チーム得点']+rot_stats['相手得点']) * 100).round(1)
print(rot_stats)

rot_stats[['自チーム得点','相手得点']].plot(kind='bar', figsize=(8,4))
plt.title('ローテーション別 得点/失点')
plt.xlabel('ローテーション')
plt.ylabel('回数')
plt.tight_layout()
plt.show()

# セル7: サーブキャッチ分析
recv = df[df['receive_grade']!='']
if len(recv)>0:
    print('=== キャッチ評価分布 ===')
    print(recv['receive_grade'].value_counts().sort_index())
    print(f'\nA率: {(recv["receive_grade"]=="A").mean()*100:.1f}%')

    # キャッチ評価別の得点率
    for g in ['A','B','C','D']:
        gd = recv[recv['receive_grade']==g]
        if len(gd)>0:
            pt = (gd['point_team']=='自チーム').mean()*100
            print(f'キャッチ{g} → 自チーム得点率: {pt:.1f}% (n={len(gd)})')

# セル8: 選手別キャッチ評価
if len(recv)>0:
    recv_by_player = pd.crosstab(recv['receiver'], recv['receive_grade'])
    print(recv_by_player)
